# LeetCode 216 — Combination Sum III bằng Python

Link đề: https://leetcode.com/problems/combination-sum-iii/ · Độ khó: **Medium** · Chủ đề: Backtracking, Array, Bit Manipulation (tổ hợp)

Skeleton LeetCode:

```python
class Solution:
    def combinationSum3(self, k: int, n: int) -> List[List[int]]:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force độc lập (dựng mọi tập con của `{1..9}` bằng cách nhân đôi dần).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mọi tập gồm đúng `k` số khác nhau từ `1..9`** có **tổng bằng `n`**; mỗi số dùng tối đa một lần, và **không được trùng tổ hợp** (`[1,2,4]` và `[2,1,4]` là **một** tổ hợp).
2. Phân biệt **tổ hợp (combination)** — không quan tâm thứ tự — với **hoán vị (permutation)**, và biết cách **ép thứ tự tăng dần** để không sinh trùng.
3. Nắm **mẫu Backtracking** (chọn → đệ quy → bỏ chọn) qua các cách: **`itertools.combinations` lọc** → **chọn/không chọn từng số** → **vòng for + cắt tỉa (pruning)** → **liệt kê mặt nạ bit (bitmask)**.
4. Hiểu cách **cắt tỉa**: dừng sớm khi tổng còn lại âm/nhỏ hơn số đang xét, hoặc không đủ số để chọn.
5. Tránh các lỗi kinh điển: **không sao chép** danh sách khi lưu kết quả (`path` vs `path[:]`), **quên bỏ chọn** (`pop`), cho **dùng lại số** (`dfs(i)` thay vì `dfs(i + 1)`), và **sinh hoán vị trùng**.

## 2. Đề bài & ví dụ

Tìm **tất cả các tổ hợp hợp lệ** gồm `k` số có tổng bằng `n`, thỏa:

- Chỉ dùng các số từ `1` đến `9`.
- **Mỗi số dùng tối đa một lần**.

Trả về **danh sách mọi tổ hợp hợp lệ**. Danh sách **không được chứa cùng một tổ hợp hai lần** và các tổ hợp có thể trả về **theo thứ tự bất kỳ**.

**Ví dụ 1**

- Input: `k = 3, n = 7`
- Output: `[[1,2,4]]`
- Giải thích: `1 + 2 + 4 = 7`; không còn tổ hợp nào khác.

**Ví dụ 2**

- Input: `k = 3, n = 9`
- Output: `[[1,2,6],[1,3,5],[2,3,4]]`
- Giải thích: `1 + 2 + 6 = 9`, `1 + 3 + 5 = 9`, `2 + 3 + 4 = 9`; không còn tổ hợp nào khác.

**Ví dụ 3**

- Input: `k = 4, n = 1`
- Output: `[]`
- Giải thích: dùng 4 số khác nhau trong `[1, 9]`, tổng nhỏ nhất là `1 + 2 + 3 + 4 = 10 > 1`, nên không có tổ hợp hợp lệ.

**Ràng buộc:** `2 <= k <= 9`; `1 <= n <= 60`.

In [1]:
from typing import List
from itertools import combinations
import random

examples = [
    (3, 7, [[1, 2, 4]]),
    (3, 9, [[1, 2, 6], [1, 3, 5], [2, 3, 4]]),
    (4, 1, []),
]
for k, n, expected in examples:
    print(f"k={k}, n={n} -> kỳ vọng {expected}")

k=3, n=7 -> kỳ vọng [[1, 2, 4]]
k=3, n=9 -> kỳ vọng [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
k=4, n=1 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Tổ hợp và hoán vị

Một **tổ hợp** là một **tập** các số, thứ tự không quan trọng: `{1, 2, 4}`. Một **hoán vị** là một **dãy**: `[1,2,4]`, `[2,1,4]`, ... là các dãy khác nhau (6 hoán vị của cùng một tổ hợp 3 số). Đề yêu cầu **không trùng tổ hợp**, nên ta quy ước **luôn chọn số theo thứ tự tăng dần**: mỗi tổ hợp được sinh đúng **một lần** dưới dạng dãy tăng.

In [2]:
print(list(combinations([1, 2, 3, 4], 2)))
# [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]: 6 tổ hợp, mỗi cặp một lần, luôn tăng dần
from itertools import permutations
print(len(list(permutations([1, 2, 3, 4], 2))), "hoán vị cùng cỡ (mỗi cặp xuất hiện 2 lần theo hai thứ tự)")

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
12 hoán vị cùng cỡ (mỗi cặp xuất hiện 2 lần theo hai thứ tự)


### 3.2. Đệ quy và Backtracking (quay lui)

**Đệ quy** là hàm tự gọi chính nó cho bài toán nhỏ hơn. **Backtracking** là đệ quy để **thử mọi lựa chọn**: ở mỗi bước ta **chọn** một số (thêm vào danh sách hiện tại `path`), **đệ quy** để chọn tiếp, rồi **bỏ chọn** (`path.pop()`) để thử lựa chọn khác. Hình dung như **cây quyết định**: mỗi nhánh là một lựa chọn, mỗi lá là một đáp án hoặc ngõ cụt.

In [3]:
def demo_dfs(start, path):
    print("  " * len(path) + f"path={path}")
    if len(path) == 2:
        return
    for d in range(start, 4):          # chọn số từ start..3
        path.append(d)                 # CHỌN
        demo_dfs(d + 1, path)          # ĐỆ QUY (số kế tiếp lớn hơn d: không dùng lại, không đảo thứ tự)
        path.pop()                     # BỎ CHỌN (quay lui)


demo_dfs(1, [])

path=[]
  path=[1]
    path=[1, 2]
    path=[1, 3]
  path=[2]
    path=[2, 3]
  path=[3]


### 3.3. Danh sách: `append`, `pop`, và sao chép

`path.append(x)` thêm cuối; `path.pop()` bỏ phần tử cuối. **Cạm bẫy lớn:** `ans.append(path)` chỉ lưu **tham chiếu** tới cùng một danh sách `path`; khi `path` thay đổi tiếp, mọi phần tử đã lưu cũng bị đổi theo. Phải lưu **bản sao**: `ans.append(path[:])` hoặc `list(path)`.

In [4]:
path, ans_bad, ans_good = [1, 2], [], []
ans_bad.append(path)
ans_good.append(path[:])
path.append(3)
print("lưu tham chiếu:", ans_bad)    # [[1, 2, 3]]  (bị đổi theo!)
print("lưu bản sao   :", ans_good)   # [[1, 2]]

lưu tham chiếu: [[1, 2, 3]]
lưu bản sao   : [[1, 2]]


### 3.4. `range(start, 10)`, `sum` và công thức tổng

Chọn số kế tiếp từ `start` tới `9`: `range(start, 10)`. Tổng `k` số **nhỏ nhất** của `1..9` là `1 + 2 + ... + k = k(k+1)/2`; tổng `k` số **lớn nhất** là `9 + 8 + ... + (10 - k) = k(19 - k)/2`. Nếu `n` nằm ngoài khoảng `[k(k+1)/2, k(19-k)/2]` thì **chắc chắn** không có đáp án — một cách cắt tỉa nhanh.

In [5]:
for k in range(2, 10):
    lo = k * (k + 1) // 2
    hi = k * (19 - k) // 2
    assert lo == sum(range(1, k + 1)) and hi == sum(range(10 - k, 10))
    print(f"k={k}: tổng nhỏ nhất {lo}, lớn nhất {hi}")

k=2: tổng nhỏ nhất 3, lớn nhất 17
k=3: tổng nhỏ nhất 6, lớn nhất 24
k=4: tổng nhỏ nhất 10, lớn nhất 30
k=5: tổng nhỏ nhất 15, lớn nhất 35
k=6: tổng nhỏ nhất 21, lớn nhất 39
k=7: tổng nhỏ nhất 28, lớn nhất 42
k=8: tổng nhỏ nhất 36, lớn nhất 44
k=9: tổng nhỏ nhất 45, lớn nhất 45


### 3.5. Mặt nạ bit (bitmask) biểu diễn tập con

Với `9` số, mỗi tập con ứng với một số nguyên `mask` từ `0` đến `2^9 - 1 = 511`: **bit thứ `i` bằng `1`** nghĩa là số `i + 1` có trong tập. `mask >> i & 1` đọc bit `i`; `mask.bit_count()` đếm số bit `1` (số phần tử của tập). Có `512` tập con, đủ nhỏ để liệt kê hết.

In [6]:
mask = 0b000000111            # bit 0, 1, 2 bật -> tập {1, 2, 3}
chosen = [i + 1 for i in range(9) if mask >> i & 1]
print(bin(mask), chosen, "số phần tử:", mask.bit_count(), "tổng:", sum(chosen))
mask = 0b100010010            # bit 1, 4, 8 -> {2, 5, 9}
print(bin(mask), [i + 1 for i in range(9) if mask >> i & 1])

0b111 [1, 2, 3] số phần tử: 3 tổng: 6
0b100010010 [2, 5, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tìm tay `k = 3, n = 9`. Ta xếp số tăng dần `a < b < c`, `a + b + c = 9`:

- `a = 1`: cần `b + c = 8` với `1 < b < c`: `(2, 6)`, `(3, 5)` (còn `(4, 4)` bị loại vì trùng). → `[1,2,6]`, `[1,3,5]`.
- `a = 2`: cần `b + c = 7` với `2 < b < c`: `(3, 4)`. → `[2,3,4]`.
- `a = 3`: cần `b + c = 6` với `3 < b < c`: tối thiểu `4 + 5 = 9 > 6`, không có.

Ba tổ hợp — khớp ví dụ 2. Cách nghĩ: **chọn số nhỏ nhất, rồi số nhỏ nhất kế tiếp, ... luôn lớn hơn số đã chọn**, cho tới khi đủ `k` số và tổng đúng bằng `n`. Mỗi lần chọn một số `d`, bài toán "lùi" thành: còn `k - 1` số, tổng còn `n - d`, các số phải `> d`. Đó là cấu trúc đệ quy của backtracking.

## 5. Cách 1 — Brute Force / baseline: sinh mọi tổ hợp `k` số rồi lọc

**Ý tưởng.** `itertools.combinations(range(1, 10), k)` sinh **mọi tổ hợp `k` số** của `1..9` (luôn tăng dần, không trùng). Giữ những tổ hợp có tổng bằng `n`. Số tổ hợp tối đa là `C(9, k) <= 126`, nhỏ.

In [7]:
def combination_sum3_filter(k, n):
    ans = []
    for combo in combinations(range(1, 10), k):
        if sum(combo) == n:
            ans.append(list(combo))            # đổi tuple thành list
    return ans


for k, n, expected in examples:
    print(k, n, "->", combination_sum3_filter(k, n), "(kỳ vọng", expected, ")")

3 7 -> [[1, 2, 4]] (kỳ vọng [[1, 2, 4]] )
3 9 -> [[1, 2, 6], [1, 3, 5], [2, 3, 4]] (kỳ vọng [[1, 2, 6], [1, 3, 5], [2, 3, 4]] )
4 1 -> [] (kỳ vọng [] )


In [8]:
# Quan sát số tổ hợp phải duyệt theo k
from math import comb
print({k: comb(9, k) for k in range(2, 10)})      # C(9,k)

{2: 36, 3: 84, 4: 126, 5: 126, 6: 84, 7: 36, 8: 9, 9: 1}


**Độ phức tạp (Cách 1).** Số tổ hợp là `C(9, k)`, mỗi tổ hợp tính tổng `O(k)`: **Time `O(C(9,k) · k)`**, tối đa `126 · 9`. Vì `9` là hằng, đó là **`O(1)`**. **Space `O(k)`** ngoài kết quả.

**Nút thắt.** Cách này **không học được gì về backtracking**: nó "dùng thư viện làm hộ". Thêm nữa, nó thử cả những tổ hợp mà ta có thể loại sớm (ví dụ khi tổng tạm thời đã vượt `n`). Ta sẽ tự dựng quá trình chọn để **cắt tỉa**.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 (thứ tự tăng):** chọn các số theo thứ tự **tăng dần**; ở bước tiếp theo chỉ được chọn số **lớn hơn số vừa chọn**. Như vậy mỗi tổ hợp xuất hiện đúng một lần (không có hoán vị trùng) và mỗi số dùng tối đa một lần.

**Quan sát 2 (bài toán con):** sau khi chọn `d`, bài toán còn lại là: chọn `k - 1` số từ `d + 1 .. 9` có tổng `n - d`. Hàm `dfs(start, remain_sum, remain_count)`.

**Quan sát 3 (cắt tỉa):** vì mọi số đều dương:

- nếu số đang xét `d > remain_sum` thì mọi số lớn hơn `d` cũng vượt tổng còn lại: **dừng vòng lặp** (`break`) ngay;
- nếu số lượng số còn lại `9 - start + 1` ít hơn `remain_count` thì không thể đủ: **dừng**;
- khi `remain_count == 0` thì kết quả đúng khi và chỉ khi `remain_sum == 0`.

Kết quả: một cây đệ quy rất nhỏ (với `k = 3, n = 7` chỉ vài chục nút).

## 7. Các cách tiếp theo

### Cách 2 — Backtracking: chọn / không chọn từng số

**Giải thích.** Với mỗi số `d = 1, 2, ..., 9` ta có **hai lựa chọn**: **lấy** `d` hoặc **bỏ** `d`. `dfs(d, remain_sum, path)` quyết định cho số `d`. Điều kiện dừng:

- `len(path) == k`: nếu `remain_sum == 0` thì ghi nhận; dừng.
- `d > 9` hoặc `remain_sum < d`: hết số / không thể nữa; dừng.

In [9]:
def combination_sum3_take_skip(k, n):
    ans, path = [], []

    def dfs(d, remain):
        if len(path) == k:                    # đủ k số
            if remain == 0:
                ans.append(path[:])           # lưu BẢN SAO
            return
        if d > 9 or remain < d:               # hết số, hoặc số nhỏ nhất còn lại đã vượt tổng cần
            return
        path.append(d)                        # lấy d
        dfs(d + 1, remain - d)
        path.pop()                            # bỏ chọn
        dfs(d + 1, remain)                    # không lấy d

    dfs(1, n)
    return ans


for k, n, expected in examples:
    print(k, n, "->", combination_sum3_take_skip(k, n), "(kỳ vọng", expected, ")")

3 7 -> [[1, 2, 4]] (kỳ vọng [[1, 2, 4]] )
3 9 -> [[1, 2, 6], [1, 3, 5], [2, 3, 4]] (kỳ vọng [[1, 2, 6], [1, 3, 5], [2, 3, 4]] )
4 1 -> [] (kỳ vọng [] )


**Độ phức tạp Cách 2:** cây có tối đa `2^9 = 512` nhánh: **Time `O(2^9 · k)`** (hằng số, nhưng thể hiện dạng `O(2^m)` với `m` số ứng viên). **Space `O(k)`** cho `path` và ngăn xếp đệ quy sâu tối đa `9`.

### Cách 3 — Backtracking bằng vòng `for` + cắt tỉa

**Giải thích.** Thay vì "lấy/bỏ", dùng vòng `for d in range(start, 10)`: ở mỗi tầng ta chọn **số kế tiếp** `d` rồi gọi `dfs(d + 1, ...)`. Vòng lặp tự sinh các nhánh. Thêm hai điều kiện cắt tỉa: `d > remain_sum` thì `break` (các số sau còn lớn hơn), và nếu số lượng số còn lại không đủ thì `return`.

In [10]:
def combination_sum3_dfs_for(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
            return
        if 9 - start + 1 < remain_count:      # không đủ số để chọn
            return
        for d in range(start, 10):
            if d > remain_sum:                # mọi số từ d trở đi đều vượt tổng còn lại
                break
            path.append(d)
            dfs(d + 1, remain_sum - d, remain_count - 1)
            path.pop()

    dfs(1, n, k)
    return ans


for k, n, expected in examples:
    print(k, n, "->", combination_sum3_dfs_for(k, n), "(kỳ vọng", expected, ")")

3 7 -> [[1, 2, 4]] (kỳ vọng [[1, 2, 4]] )
3 9 -> [[1, 2, 6], [1, 3, 5], [2, 3, 4]] (kỳ vọng [[1, 2, 6], [1, 3, 5], [2, 3, 4]] )
4 1 -> [] (kỳ vọng [] )


In [11]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def combinationSum3(self, k: int, n: int) -> List[List[int]]:
        ans, path = [], []

        def dfs(start: int, remain_sum: int, remain_count: int) -> None:
            if remain_count == 0:
                if remain_sum == 0:
                    ans.append(path[:])
                return
            if 9 - start + 1 < remain_count:
                return
            for d in range(start, 10):
                if d > remain_sum:
                    break
                path.append(d)
                dfs(d + 1, remain_sum - d, remain_count - 1)
                path.pop()

        dfs(1, n, k)
        return ans


print(Solution().combinationSum3(3, 9))   # [[1, 2, 6], [1, 3, 5], [2, 3, 4]]

[[1, 2, 6], [1, 3, 5], [2, 3, 4]]


**Độ phức tạp Cách 3:** **Time `O(C(9, k) · k)`** trong worst case (tối đa lần lượt thăm các tổ hợp đủ `k` số, nhưng cắt tỉa loại rất nhiều nhánh), **Space `O(k)`** (ngăn xếp và `path`). Vì `9` cố định nên là hằng số.

### Cách 4 — Liệt kê mặt nạ bit (bitmask)

**Giải thích.** Duyệt `mask` từ `0` tới `511`. Giữ `mask` nếu có đúng `k` bit `1` (`bit_count() == k`); lấy các số tương ứng và kiểm tra tổng bằng `n`. Cách này không đệ quy, rất ngắn, đáng nhớ khi tập ứng viên nhỏ (`<= 20` phần tử).

In [12]:
def combination_sum3_bitmask(k, n):
    ans = []
    for mask in range(1 << 9):
        if bin(mask).count("1") != k:                         # (3.10+ có thể dùng mask.bit_count())
            continue
        chosen = [i + 1 for i in range(9) if mask >> i & 1]   # số được chọn, tăng dần
        if sum(chosen) == n:
            ans.append(chosen)
    return ans


for k, n, expected in examples:
    print(k, n, "->", combination_sum3_bitmask(k, n), "(kỳ vọng", expected, ")")

3 7 -> [[1, 2, 4]] (kỳ vọng [[1, 2, 4]] )
3 9 -> [[2, 3, 4], [1, 3, 5], [1, 2, 6]] (kỳ vọng [[1, 2, 6], [1, 3, 5], [2, 3, 4]] )
4 1 -> [] (kỳ vọng [] )


**Độ phức tạp Cách 4:** **Time `O(2^9 · 9)`** (mỗi mask đọc `9` bit) — hằng số; tổng quát `O(2^m · m)` với `m` số ứng viên. **Space `O(k)`** ngoài kết quả.

> Cách 3 là cách nộp tiêu chuẩn của dạng bài "tổ hợp có tổng": mẫu chọn–đệ quy–bỏ chọn dùng được cho Combination Sum I, II, IV và nhiều bài khác.

## 8. TRACE từng bước

Ta trace Cách 3 cho `k = 3, n = 7`: mỗi lần vào `dfs` in `path`, `start`, `remain_sum`, `remain_count` và lý do dừng.

In [13]:
def trace_dfs(k, n):
    ans, path = [], []
    calls = 0

    def dfs(start, remain_sum, remain_count):
        nonlocal calls
        calls += 1
        pad = "  " * len(path)
        print(f"{pad}dfs(start={start}, remain_sum={remain_sum}, remain_count={remain_count}), path={path}")
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
                print(f"{pad}  => TÌM THẤY {path}")
            else:
                print(f"{pad}  => đủ số nhưng tổng còn dư {remain_sum}, bỏ")
            return
        if 9 - start + 1 < remain_count:
            print(f"{pad}  => không đủ số để chọn, cắt tỉa")
            return
        for d in range(start, 10):
            if d > remain_sum:
                print(f"{pad}  => d={d} > remain_sum={remain_sum}, break")
                break
            path.append(d)
            dfs(d + 1, remain_sum - d, remain_count - 1)
            path.pop()

    dfs(1, n, k)
    print("số lần gọi dfs:", calls, "| kết quả:", ans)
    return ans


trace_dfs(3, 7)

dfs(start=1, remain_sum=7, remain_count=3), path=[]
  dfs(start=2, remain_sum=6, remain_count=2), path=[1]
    dfs(start=3, remain_sum=4, remain_count=1), path=[1, 2]
      dfs(start=4, remain_sum=1, remain_count=0), path=[1, 2, 3]
        => đủ số nhưng tổng còn dư 1, bỏ
      dfs(start=5, remain_sum=0, remain_count=0), path=[1, 2, 4]
        => TÌM THẤY [1, 2, 4]
      => d=5 > remain_sum=4, break
    dfs(start=4, remain_sum=3, remain_count=1), path=[1, 3]
      => d=4 > remain_sum=3, break
    dfs(start=5, remain_sum=2, remain_count=1), path=[1, 4]
      => d=5 > remain_sum=2, break
    dfs(start=6, remain_sum=1, remain_count=1), path=[1, 5]
      => d=6 > remain_sum=1, break
    dfs(start=7, remain_sum=0, remain_count=1), path=[1, 6]
      => d=7 > remain_sum=0, break
    => d=7 > remain_sum=6, break
  dfs(start=3, remain_sum=5, remain_count=2), path=[2]
    dfs(start=4, remain_sum=2, remain_count=1), path=[2, 3]
      => d=4 > remain_sum=2, break
    dfs(start=5, remain_sum=1, rem

[[1, 2, 4]]

Đọc trace: nhánh đầu `path = [1]`, rồi `[1,2]`; ở tầng cuối `d = 3` cho `remain_sum = 1` (đủ số nhưng dư `1`, bỏ), `d = 4` cho `remain_sum = 0` nên `[1,2,4]` được ghi nhận, `d = 5 > 4` thì `break`. Các nhánh `[1,3,...]`, `[1,4,...]`, ... không cho tổng đúng 7 với đủ 3 số. Từ `path = [2]` trở đi: cần thêm 2 số lớn hơn `2` có tổng `5`, nhưng nhỏ nhất là `3 + 4 = 7 > 5` nên không có nghiệm.

In [14]:
# Bảng tóm tắt: số lần gọi dfs với/không cắt tỉa (đếm nút của cây đệ quy)
def count_nodes(k, n, prune):
    calls = 0

    def dfs(start, remain_sum, remain_count):
        nonlocal calls
        calls += 1
        if remain_count == 0:
            return
        if prune and 9 - start + 1 < remain_count:
            return
        for d in range(start, 10):
            if prune and d > remain_sum:
                break
            dfs(d + 1, remain_sum - d, remain_count - 1)

    dfs(1, n, k)
    return calls


for k, n in [(3, 7), (3, 9), (4, 1), (5, 25), (9, 45)]:
    print(f"k={k}, n={n}: không cắt tỉa {count_nodes(k, n, False):>4} nút | có cắt tỉa {count_nodes(k, n, True):>4} nút")

k=3, n=7: không cắt tỉa  130 nút | có cắt tỉa   19 nút
k=3, n=9: không cắt tỉa  130 nút | có cắt tỉa   33 nút
k=4, n=1: không cắt tỉa  256 nút | có cắt tỉa    2 nút
k=5, n=25: không cắt tỉa  382 nút | có cắt tỉa  276 nút
k=9, n=45: không cắt tỉa  512 nút | có cắt tỉa   46 nút


## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (vòng `for` + cắt tỉa). Đúng `class Solution`, đúng `combinationSum3(self, k: int, n: int) -> List[List[int]]`.

In [15]:
class Solution:
    def combinationSum3(self, k: int, n: int) -> List[List[int]]:
        ans: List[List[int]] = []
        path: List[int] = []

        def dfs(start: int, remain_sum: int, remain_count: int) -> None:
            if remain_count == 0:
                if remain_sum == 0:
                    ans.append(path[:])          # lưu bản sao, không lưu tham chiếu
                return
            if 9 - start + 1 < remain_count:     # không đủ số còn lại
                return
            for d in range(start, 10):
                if d > remain_sum:               # các số sau còn lớn hơn: dừng
                    break
                path.append(d)                   # chọn
                dfs(d + 1, remain_sum - d, remain_count - 1)   # d + 1: không dùng lại, giữ thứ tự tăng
                path.pop()                       # bỏ chọn

        dfs(1, n, k)
        return ans


sol = Solution()
for k, n, expected in examples:
    assert sorted(sol.combinationSum3(k, n)) == sorted(expected)
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Mỗi tổ hợp đúng một lần:** ta chọn theo thứ tự **tăng ngặt** (`dfs(d + 1, ...)`), nên mỗi tập `k` số tương ứng với **đúng một** đường đi từ gốc (số nhỏ nhất, số nhỏ thứ hai, ...). Không có hoán vị trùng, không dùng lại số.
- **Không bỏ sót:** vòng `for` thử mọi `d` từ `start` tới `9` ở mỗi tầng; các `break`/`return` chỉ loại những nhánh **chắc chắn vô nghiệm** (`d > remain_sum` kéo theo mọi số lớn hơn cũng vượt vì các số dương; thiếu số để chọn đủ `k`).
- **Điều kiện ghi nhận đúng:** khi `remain_count == 0` ta đã chọn đúng `k` số; tổng bằng `n` khi và chỉ khi `remain_sum == 0`.
- **Backtracking đúng trạng thái:** mỗi `path.append(d)` đều có `path.pop()` tương ứng sau lời gọi đệ quy, nên khi quay về tầng trên `path` trở lại đúng như lúc vào vòng lặp (invariant).
- **Kết quả sao chép:** `path[:]` đảm bảo các đáp án đã lưu không bị đổi theo `path` về sau.

## 11. Phân tích Time / Space Complexity

Gọi `m = 9` là số ứng viên (cố định). Số kết quả tối đa là `C(9, k) <= 126`.

- **Cách 1 (`combinations` + lọc):** Time `O(C(m,k)·k)`, Space `O(k)`.
- **Cách 2 (lấy/bỏ):** Time `O(2^m · k)` trong worst case (duyệt cả cây nhị phân), Space `O(m)` ngăn xếp.
- **Cách 3 (vòng `for` + cắt tỉa):** Time `O(C(m,k)·k)` trong worst case, thường nhỏ hơn nhiều nhờ cắt tỉa; Space `O(k)`.
- **Cách 4 (bitmask):** Time `O(2^m · m)`, Space `O(k)`.

Vì `m = 9` là hằng số, cả bốn cách đều là **`O(1)`** theo nghĩa chặt (tối đa vài nghìn phép). Ta vẫn viết theo `m` để thấy tốc độ tăng nếu đề mở rộng ứng viên lên `1..m` lớn hơn: khi đó `O(2^m)` bùng nổ, còn cắt tỉa giúp rất nhiều. Không gian tổng kể cả đầu ra là `O(C(m,k)·k)`.

## 12. Bộ test

`oracle` là brute force **độc lập**: dựng **mọi tập con** của `{1..9}` bằng cách nhân đôi dần (mỗi số mới `d` thêm vào mọi tập con cũ), rồi lọc theo kích thước và tổng. Ta so sánh sau khi chuẩn hóa (sắp xếp) vì đề cho phép trả theo thứ tự bất kỳ.

In [16]:
def oracle(k, n):
    """Brute force độc lập: dựng mọi tập con của {1..9} bằng cách nhân đôi dần."""
    subsets = [()]
    for d in range(1, 10):
        subsets = subsets + [s + (d,) for s in subsets]
    return sorted(list(s) for s in subsets if len(s) == k and sum(s) == n)


def normalize(result):
    return sorted(sorted(c) for c in result)


impls = {
    "filter": combination_sum3_filter,
    "take_skip": combination_sum3_take_skip,
    "dfs_for": combination_sum3_dfs_for,
    "bitmask": combination_sum3_bitmask,
    "submit": sol.combinationSum3,
}

for k, n, expected in examples:
    assert oracle(k, n) == sorted(expected)
    for key, fn in impls.items():
        assert normalize(fn(k, n)) == sorted(expected), (key, k, n, fn(k, n), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [17]:
edge_cases = [
    (2, 3, [[1, 2]], "tổng nhỏ nhất của k=2"),
    (2, 17, [[8, 9]], "tổng lớn nhất của k=2"),
    (2, 18, [], "k=2 vượt tổng lớn nhất"),
    (9, 45, [[1, 2, 3, 4, 5, 6, 7, 8, 9]], "k=9 dùng đủ 9 số"),
    (9, 44, [], "k=9 mà tổng khác 45"),
    (3, 24, [[7, 8, 9]], "k=3 tổng lớn nhất"),
    (3, 6, [[1, 2, 3]], "k=3 tổng nhỏ nhất"),
    (3, 5, [], "k=3 nhỏ hơn tổng nhỏ nhất"),
    (2, 60, [], "n lớn nhất của đề"),
    (4, 10, [[1, 2, 3, 4]], "k=4 tổng nhỏ nhất"),
]
for k, n, expected, label in edge_cases:
    assert oracle(k, n) == sorted(expected), (label, oracle(k, n), expected)
    for key, fn in impls.items():
        assert normalize(fn(k, n)) == sorted(expected), (key, label, k, n, fn(k, n), expected)
    print("PASS", label)

PASS tổng nhỏ nhất của k=2
PASS tổng lớn nhất của k=2
PASS k=2 vượt tổng lớn nhất
PASS k=9 dùng đủ 9 số
PASS k=9 mà tổng khác 45
PASS k=3 tổng lớn nhất
PASS k=3 tổng nhỏ nhất
PASS k=3 nhỏ hơn tổng nhỏ nhất
PASS n lớn nhất của đề
PASS k=4 tổng nhỏ nhất


### 12.2. Tính chất và phản ví dụ

- **Tổng số đáp án** qua mọi `(k, n)` với `2 <= k <= 9`: mỗi tập con của `{1..9}` có kích thước `>= 2` ứng với đúng một cặp `(k, n)`, nên tổng bằng `2^9 - 1 - 9 = 502`.
- Mỗi đáp án **tăng ngặt**, đúng `k` phần tử, thuộc `[1, 9]`, và tổng bằng `n`.
- Đối xứng: phép thay `x -> 10 - x` biến một tổ hợp `k` số tổng `n` thành tổ hợp `k` số tổng `10k - n`; số đáp án của `(k, n)` và `(k, 10k - n)` bằng nhau.
- **Phản ví dụ** cho lỗi hoán vị trùng: vòng `for` luôn bắt đầu từ `1` sinh cả `[1,2,4]`, `[2,1,4]`, ...

In [18]:
total = 0
for k in range(2, 10):
    for n in range(1, 61):
        res = sol.combinationSum3(k, n)
        total += len(res)
        for c in res:
            assert len(c) == k and sum(c) == n and all(1 <= x <= 9 for x in c)
            assert all(c[i] < c[i + 1] for i in range(len(c) - 1))
        assert len(res) == len(sol.combinationSum3(k, 10 * k - n)) if 1 <= 10 * k - n <= 60 else True
print("tổng số đáp án qua mọi (k, n):", total)
assert total == 502
print("PASS tính chất (tổng 502, tăng ngặt, đối xứng 10 - x)")


def sai_hoan_vi(k, n):
    ans, path = [], []

    def dfs(remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
            return
        for d in range(1, 10):                      # SAI: luôn bắt đầu từ 1
            if d in path:
                continue
            path.append(d)
            dfs(remain_sum - d, remain_count - 1)
            path.pop()

    dfs(n, k)
    return ans


print("phản ví dụ k=3, n=7: đúng", sol.combinationSum3(3, 7), "| hoán vị:", sai_hoan_vi(3, 7))
assert len(sai_hoan_vi(3, 7)) == 6 and len(sol.combinationSum3(3, 7)) == 1

tổng số đáp án qua mọi (k, n): 502
PASS tính chất (tổng 502, tăng ngặt, đối xứng 10 - x)
phản ví dụ k=3, n=7: đúng [[1, 2, 4]] | hoán vị: [[1, 2, 4], [1, 4, 2], [2, 1, 4], [2, 4, 1], [4, 1, 2], [4, 2, 1]]


### 12.3. Đối chiếu toàn bộ (miền rất nhỏ nên vét cạn)

Miền đầu vào chỉ có `8 × 60 = 480` cặp `(k, n)` — ta kiểm tra **mọi** cặp, thêm một đợt ngẫu nhiên có seed cho đủ quy trình.

In [19]:
for k in range(2, 10):
    for n in range(1, 61):
        truth = oracle(k, n)
        for key, fn in impls.items():
            assert normalize(fn(k, n)) == truth, (key, k, n)
print("PASS vét cạn 480 cặp (k, n) trong miền của đề: mọi cách khớp oracle")

random.seed(216)
for _ in range(500):
    k, n = random.randint(2, 9), random.randint(1, 60)
    assert normalize(sol.combinationSum3(k, n)) == oracle(k, n)
print("PASS 500 cặp ngẫu nhiên có seed")

PASS vét cạn 480 cặp (k, n) trong miền của đề: mọi cách khớp oracle


PASS 500 cặp ngẫu nhiên có seed


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không sao chép `path` khi lưu kết quả
#### Code sai

In [20]:
def sai_khong_copy(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path)                     # SAI: lưu tham chiếu
            return
        for d in range(start, 10):
            if d > remain_sum:
                break
            path.append(d)
            dfs(d + 1, remain_sum - d, remain_count - 1)
            path.pop()

    dfs(1, n, k)
    return ans


print("đúng:", Solution().combinationSum3(3, 9), "| sai:", sai_khong_copy(3, 9))

đúng: [[1, 2, 6], [1, 3, 5], [2, 3, 4]] | sai: [[], [], []]


**Vì sao sai:** `ans` chứa **cùng một đối tượng** `path`; cuối cùng `path` rỗng nên ta được `[[], [], []]`. **Code đúng:** `ans.append(path[:])`.

### Lỗi 2: Quên bỏ chọn (`pop`)
#### Code sai

In [21]:
def sai_quen_pop(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
            return
        for d in range(start, 10):
            if d > remain_sum:
                break
            path.append(d)
            dfs(d + 1, remain_sum - d, remain_count - 1)
            # SAI: thiếu path.pop()

    dfs(1, n, k)
    return ans


print("đúng:", Solution().combinationSum3(3, 7), "| sai:", sai_quen_pop(3, 7))

đúng: [[1, 2, 4]] | sai: [[1, 2, 3, 4]]


**Vì sao sai:** `path` không được trả về trạng thái cũ sau khi thử xong một nhánh, nên các nhánh sau bị "dính" số cũ và `remain_count` không khớp độ dài `path`. **Code đúng:** mỗi `append` đi kèm một `pop`.

### Lỗi 3: Cho dùng lại số (`dfs(d)` thay vì `dfs(d + 1)`)
#### Code sai

In [22]:
def sai_dung_lai(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
            return
        for d in range(start, 10):
            if d > remain_sum:
                break
            path.append(d)
            dfs(d, remain_sum - d, remain_count - 1)      # SAI: phải là d + 1
            path.pop()

    dfs(1, n, k)
    return ans


print("đúng:", Solution().combinationSum3(3, 9), "| sai:", sai_dung_lai(3, 9))

đúng: [[1, 2, 6], [1, 3, 5], [2, 3, 4]] | sai: [[1, 1, 7], [1, 2, 6], [1, 3, 5], [1, 4, 4], [2, 2, 5], [2, 3, 4], [3, 3, 3]]


**Vì sao sai:** đề cho dùng **mỗi số tối đa một lần**; `dfs(d, ...)` cho phép chọn lại `d` nên có `[1,1,7]`, `[3,3,3]`,... **Code đúng:** `dfs(d + 1, ...)`.

### Lỗi 4: Sinh hoán vị trùng
Xem `sai_hoan_vi` ở phần test: bắt đầu vòng `for` từ `1` ở mọi tầng sinh `6` hoán vị của `[1,2,4]` thay vì `1` tổ hợp. **Code đúng:** truyền `start` để chỉ chọn số **lớn hơn** số vừa chọn.

### Lỗi 5: Bỏ cắt tỉa và đếm sai điều kiện dừng
Nếu chỉ kiểm tra `remain_sum == 0` mà không kiểm tra `remain_count == 0`, ta nhận cả tổ hợp có **sai số lượng** (ví dụ `k = 3, n = 7` nhận `[1, 6]` hay `[3, 4]`).
#### Code sai

In [23]:
def sai_thieu_dem(k, n):
    ans, path = [], []

    def dfs(start, remain_sum):
        if remain_sum == 0:                              # SAI: không kiểm tra len(path) == k
            ans.append(path[:])
            return
        for d in range(start, 10):
            if d > remain_sum:
                break
            path.append(d)
            dfs(d + 1, remain_sum - d)
            path.pop()

    dfs(1, n)
    return ans


print("đúng:", Solution().combinationSum3(3, 7), "| sai:", sai_thieu_dem(3, 7))

đúng: [[1, 2, 4]] | sai: [[1, 2, 4], [1, 6], [2, 5], [3, 4], [7]]


**Vì sao sai:** đề yêu cầu **đúng `k` số**. **Code đúng:** ghi nhận khi `remain_count == 0` **và** `remain_sum == 0`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**tìm tất cả** tổ hợp / tập con / cách chọn thỏa điều kiện" → **Backtracking**. Mẫu: `chọn → đệ quy → bỏ chọn`.
- Thấy "**không trùng** tổ hợp", "mỗi phần tử dùng **một lần**" → truyền `start = d + 1`; nếu có phần tử trùng trong đầu vào (Combination Sum II) thì thêm bước **bỏ qua giá trị lặp** ở cùng tầng.
- Thấy "**dùng lại** được" (Combination Sum I) → đệ quy với `start = d` (không cộng 1).
- Muốn tăng tốc: **sắp xếp** ứng viên rồi `break` khi `d` vượt tổng còn lại; **tính cận trên/dưới** của tổng (`k(k+1)/2` và `k(19-k)/2`).
- Ứng viên **rất ít** (`<= 20`) → liệt kê **bitmask** `2^m` là cách ngắn và ít lỗi.
- Lưu ý khi lưu kết quả: **`path[:]`**. Đây là lỗi đứng đầu bảng của người mới.
- Tự kiểm tra bằng ví dụ **không có đáp án** (`k=4, n=1`) và ví dụ **dùng đủ 9 số** (`k=9, n=45`).

## 15. Biến thể của bài

1. **Combination Sum** (39): số được **dùng lại** vô hạn lần, danh sách ứng viên cho trước.
2. **Combination Sum II** (40): ứng viên có **trùng**, mỗi phần tử dùng một lần, kết quả không trùng.
3. **Combination Sum IV** (377): **đếm** số cách (có thứ tự) — bài quy hoạch động.
4. **Combinations** (77): chọn `k` số từ `1..n` (không ràng buộc tổng).
5. **Subsets** (78), **Permutations** (46): cùng khung backtracking.
6. **Đếm** số tổ hợp của Combination Sum III (không liệt kê): quy hoạch động theo `(số, số lượng, tổng)`.

In [24]:
def combinations_k_of_n(n, k):
    """LeetCode 77: chọn k số từ 1..n."""
    ans, path = [], []

    def dfs(start):
        if len(path) == k:
            ans.append(path[:])
            return
        for d in range(start, n + 1):
            path.append(d)
            dfs(d + 1)
            path.pop()

    dfs(1)
    return ans


def count_combination_sum3(k, n):
    """Đếm (không liệt kê) bằng DP: dp[c][s] = số cách chọn c số có tổng s từ các số đã xét."""
    dp = [[0] * (n + 1) for _ in range(k + 1)]
    dp[0][0] = 1
    for d in range(1, 10):
        for c in range(k, 0, -1):                 # duyệt ngược để mỗi số dùng tối đa một lần
            for s in range(n, d - 1, -1):
                dp[c][s] += dp[c - 1][s - d]
    return dp[k][n]


print(combinations_k_of_n(4, 2))
print([count_combination_sum3(3, n) for n in (7, 9, 15)], [len(Solution().combinationSum3(3, n)) for n in (7, 9, 15)])

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
[1, 3, 8] [1, 3, 8]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `combinations` + lọc | sinh mọi tổ hợp `k` số rồi lọc theo tổng | `O(C(9,k)·k)` | `O(k)` | nhanh để kiểm tra, không học được backtracking |
| 2. Lấy / bỏ từng số | cây nhị phân quyết định | `O(2^9·k)` | `O(9)` | giải thích rõ nhất, dạng tổng quát "mỗi phần tử chọn hoặc không" |
| 3. Vòng `for` + cắt tỉa | chọn số kế tiếp lớn hơn, `break` khi vượt tổng | `O(C(9,k)·k)` worst | `O(k)` | **cách nộp chuẩn** |
| 4. Bitmask | duyệt `2^9` mặt nạ, lọc theo số bit và tổng | `O(2^9·9)` | `O(k)` | tập ứng viên nhỏ, muốn tránh đệ quy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `list(combinations([1, 2, 3], 2))`; (b) `bin(5).count("1")`; (c) `[i + 1 for i in range(4) if 0b1010 >> i & 1]`; (d) `Solution().combinationSum3(2, 17)`.

**Bài 2 (tính bằng tay).** Liệt kê mọi tổ hợp của `k = 2, n = 10` và `k = 4, n = 14`.

**Bài 3 (điền TODO).** Hoàn thiện `combo_todo(k, n)` (Cách 3).

In [25]:
def combo_todo(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        # TODO 1: nếu remain_count == 0: nếu remain_sum == 0 thì ans.append(path[:]); rồi return
        # TODO 2: duyệt d từ start tới 9
        # TODO 3: nếu d > remain_sum thì break
        # TODO 4: path.append(d), gọi dfs(d + 1, remain_sum - d, remain_count - 1), rồi path.pop()
        pass

    # TODO 5: gọi dfs(1, n, k) rồi trả ans
    pass


# Tự làm xong rồi mới mở test:
# for k, n, expected in examples:
#     assert sorted(combo_todo(k, n)) == sorted(expected)

**Bài 4 (tự viết lại).** Viết `combo_take_skip2(k, n)` theo kiểu **lấy / bỏ** từng số `d = 1..9`, **không nhìn lại** Cách 2.

**Bài 5 (biến thể / mini challenge).** Viết `combo_sum_candidates(candidates, target)` (LeetCode 39): mỗi số trong `candidates` (toàn số dương, khác nhau) có thể được dùng **nhiều lần**; trả mọi tổ hợp có tổng `target`. Thử `candidates = [2, 3, 6, 7]`, `target = 7` → `[[2,2,3],[7]]`. Chỉ cần đổi **một** chỗ so với Cách 3.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `combinations` không lặp và giữ thứ tự gốc; `bin(5) = '0b101'`; `0b1010` có bit `1` và bit `3` bật; tổng lớn nhất của `k = 2` là `8 + 9`. Bài 2: chọn số nhỏ nhất rồi số lớn hơn. Bài 5: xét "dùng lại" thì gọi `dfs` với `start = i` (không phải `i + 1`).
- **Mức 2 (vừa):** Bài 1: `[(1,2),(1,3),(2,3)]`, `2`, `[2, 4]`, `[[8,9]]`. Bài 2: `k=2, n=10`: `[1,9],[2,8],[3,7],[4,6]`; `k=4, n=14`: bắt đầu với `1,2,3,8`... hãy lập bảng theo số nhỏ nhất. Bài 3: `if remain_count == 0: ...; return`, `for d in range(start, 10)`.
- **Mức 3 (sâu):** Bài 2: `k=4, n=14`: `[1,2,3,8]`, `[1,2,4,7]`, `[1,2,5,6]`, `[1,3,4,6]`, `[2,3,4,5]`. Bài 4: `dfs(d, remain)` với hai lời gọi `dfs(d+1, remain-d)` (sau khi `append`) và `dfs(d+1, remain)` (sau khi `pop`). Bài 5: `dfs(i, remain - candidates[i])` thay cho `dfs(i + 1, ...)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert list(combinations([1, 2, 3], 2)) == [(1, 2), (1, 3), (2, 3)]
assert bin(5).count("1") == 2
assert [i + 1 for i in range(4) if 0b1010 >> i & 1] == [2, 4]
assert Solution().combinationSum3(2, 17) == [[8, 9]]

# Bài 2
assert oracle(2, 10) == [[1, 9], [2, 8], [3, 7], [4, 6]]
assert oracle(4, 14) == [[1, 2, 3, 8], [1, 2, 4, 7], [1, 2, 5, 6], [1, 3, 4, 6], [2, 3, 4, 5]]


# Bài 3
def combo_todo_ans(k, n):
    ans, path = [], []

    def dfs(start, remain_sum, remain_count):
        if remain_count == 0:
            if remain_sum == 0:
                ans.append(path[:])
            return
        for d in range(start, 10):
            if d > remain_sum:
                break
            path.append(d)
            dfs(d + 1, remain_sum - d, remain_count - 1)
            path.pop()

    dfs(1, n, k)
    return ans


for k, n, expected in examples:
    assert sorted(combo_todo_ans(k, n)) == sorted(expected)


# Bài 4
def combo_take_skip2(k, n):
    ans, path = [], []

    def dfs(d, remain):
        if len(path) == k:
            if remain == 0:
                ans.append(path[:])
            return
        if d > 9 or remain < d:
            return
        path.append(d)
        dfs(d + 1, remain - d)
        path.pop()
        dfs(d + 1, remain)

    dfs(1, n)
    return ans


for k, n, expected in examples:
    assert sorted(combo_take_skip2(k, n)) == sorted(expected)


# Bài 5
def combo_sum_candidates(candidates, target):
    candidates = sorted(candidates)
    ans, path = [], []

    def dfs(i, remain):
        if remain == 0:
            ans.append(path[:])
            return
        for j in range(i, len(candidates)):
            if candidates[j] > remain:
                break
            path.append(candidates[j])
            dfs(j, remain - candidates[j])          # j (không phải j + 1): được dùng lại
            path.pop()

    dfs(0, target)
    return ans


assert combo_sum_candidates([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (backtracking chọn số tăng dần + cắt tỉa)

```python
class Solution:
    def combinationSum3(self, k: int, n: int) -> List[List[int]]:
        ans, path = [], []

        def dfs(start, remain_sum, remain_count):
            if remain_count == 0:
                if remain_sum == 0:
                    ans.append(path[:])
                return
            if 9 - start + 1 < remain_count:
                return
            for d in range(start, 10):
                if d > remain_sum:
                    break
                path.append(d)
                dfs(d + 1, remain_sum - d, remain_count - 1)
                path.pop()

        dfs(1, n, k)
        return ans
```

**Checklist**

- [ ] Chọn số theo thứ tự **tăng dần** (`dfs(d + 1, ...)`) để không dùng lại số và không sinh hoán vị trùng?
- [ ] Kiểm tra **đủ `k` số** và **tổng đúng `n`** mới ghi nhận?
- [ ] Mỗi `append` có `pop` tương ứng (bỏ chọn)?
- [ ] Lưu kết quả bằng `path[:]` (bản sao)?
- [ ] Cắt tỉa: `d > remain_sum` thì `break`; không đủ số còn lại thì `return`?
- [ ] Đã thử `k=4, n=1` (rỗng), `k=9, n=45` (một đáp án), `k=2, n=17`?
- [ ] Đã vét cạn / đối chiếu với brute force độc lập trên mọi `(k, n)`?
- [ ] Skeleton đúng `class Solution` / `def combinationSum3(self, k: int, n: int) -> List[List[int]]`?